#### 0.Labels
We will setup the environment for our task2 in this step.

In [1]:
from pathlib import Path
ROOT = Path.cwd().resolve().parents[1]
MATRIX = ROOT / "task1_revision" / "data" / "matrix_5min"
OUT = ROOT / "task2" / "data"
assert MATRIX.exists(), MATRIX

In [2]:
from pyspark.sql import SparkSession, functions as F, Window

spark = (SparkSession.builder
         .master("local[8]")                                     # Runing on 8 cores of the local machine
         .appName("task2-labels")
         .config("spark.driver.memory", "10g")                   
         .config("spark.sql.session.timeZone", "Asia/Singapore") 
         .config("spark.sql.shuffle.partitions", "96")           # a number slightly larger the that of stations
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/11 15:48:10 WARN Utils: Your hostname, Lynns-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.12 instead (on interface en0)
26/10/11 15:48:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/11 15:48:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [19]:
from datetime import datetime
from zoneinfo import ZoneInfo
ORIGIN_EPOCH = int(datetime(2017, 1, 1, tzinfo=ZoneInfo("Asia/Singapore")).timestamp())

# Now read the parquet files
reader = spark.read.option("pathGlobFilter", "*.parquet")
wide = reader.parquet(str(MATRIX))
wide.show(5)

+------+----+---+---+----+----+----+----+----+----+----+----+----+----+---+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+---+----+---+---+---+----+---+---+---+---+---+----+----+---+----+---+---+---+---+----+---+---+---+---+---+---+---+---+---+----+---+---+---+----+----+-----------+
|  slot| S06|S07|S08|S100|S101|S102|S103|S104|S105|S106|S107|S108|S109|S11|S110|S111|S112|S113|S114|S115|S116|S117|S118|S119|S120|S121|S122|S123|S201|S202|S203|S204|S205|S207|S208|S209|S210|S211|S212|S213|S214|S215|S216|S217|S218|S219|S220|S221|S222|S223|S224|S226|S227|S228|S229|S230|S24|S24B|S29|S33|S35| S36|S40|S43|S44|S46|S50| S55| S60|S61| S63|S64|S66|S69|S71| S72|S77|S78|S79|S81|S82|S84|S88|S89|S90|S900|S91|S92|S94| S96| S97|source_year|
+------+----+---+---+----+----+----+----+----+----+----+----+----+----+---+----+----+----+----+----+----+-

In [ ]:
import pandas as pd
stations_ids = pd.read_csv(MATRIX / "stations.csv").station_id.tolist()
long = wide.unpivot(
    ids=wide.slot,
    values=
)

+------+----+---+---+----+----+----+----+----+----+----+----+----+----+---+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+---+----+---+---+---+----+---+---+---+---+---+----+----+---+----+---+---+---+---+----+---+---+---+---+---+---+---+---+---+----+---+---+---+----+----+-----------+
|  slot| S06|S07|S08|S100|S101|S102|S103|S104|S105|S106|S107|S108|S109|S11|S110|S111|S112|S113|S114|S115|S116|S117|S118|S119|S120|S121|S122|S123|S201|S202|S203|S204|S205|S207|S208|S209|S210|S211|S212|S213|S214|S215|S216|S217|S218|S219|S220|S221|S222|S223|S224|S226|S227|S228|S229|S230|S24|S24B|S29|S33|S35| S36|S40|S43|S44|S46|S50| S55| S60|S61| S63|S64|S66|S69|S71| S72|S77|S78|S79|S81|S82|S84|S88|S89|S90|S900|S91|S92|S94| S96| S97|source_year|
+------+----+---+---+----+----+----+----+----+----+----+----+----+----+---+----+----+----+----+----+----+-